In [1]:
import pandas as pd
import numpy as np
import datetime
from epiweeks import Week
import datetime
import random
from datetime import timedelta
import requests
import io
import os

import warnings
warnings.filterwarnings('ignore')

# 2020--2024
https://github.com/reichlab/covid19-forecast-hub/tree/master

In [2]:
def pull_covid_early_predictions(model,date):
    """Load predictions of the model saved by the Covid forecast hub

    Parameters
    ----------
    model : str
        Model name on the
    dates : list or string
        List of potential dates in the iso format, e.g., 'yyyy-mm-dd', for the submission.
    """
    predictions = None
    
    url = f"https://raw.githubusercontent.com/reichlab/covid19-forecast-hub/refs/heads/master/data-processed//{model}/{date}-{model}"
    for ext in [".csv",".gz",".zip",".csv.zip",".csv.gz", '.parquet']:
        try:
            if ext == '.parquet':
                predictions = pd.read_parquet(url+ext,engine='auto')
                predictions['location'] = predictions['location'].astype(str)
                predictions['target_end_date'] = pd.to_datetime(predictions['target_end_date'])
            else:
                predictions = pd.read_csv(url+ext,dtype={'location':str},parse_dates=['target_end_date'])
        except:
            pass
    if predictions is None:
        print(f"Data for model {model} and date {date} unavailable")
    return predictions


In [ ]:
dates = ['2020-04-13', '2020-04-27', '2020-05-04', '2020-05-11', '2020-05-18', '2020-05-25', '2020-06-01', 
         '2020-06-08', '2020-06-15', '2020-06-22', '2020-06-29', '2020-07-06', '2020-07-13', '2020-07-20', 
         '2020-07-27', '2020-08-03', '2020-08-10', '2020-08-17', '2020-08-24', '2020-08-31', '2020-09-07', 
         '2020-09-14', '2020-09-21', '2020-09-28', '2020-10-05', '2020-10-12', '2020-10-19', '2020-10-26', 
         '2020-11-02', '2020-11-09', '2020-11-16', '2020-11-23', '2020-11-30', '2020-12-07', '2020-12-14',
         '2020-12-21', '2020-12-28', '2021-01-04', '2021-01-11', '2021-01-18', '2021-01-25', '2021-02-01', 
         '2021-02-08', '2021-02-15', '2021-02-22', '2021-03-01', '2021-03-08', '2021-03-15', '2021-03-22', 
         '2021-03-29', '2021-04-05', '2021-04-12', '2021-04-19', '2021-04-26', '2021-05-03', '2021-05-10', 
         '2021-05-17', '2021-05-24', '2021-05-31', '2021-06-07', '2021-06-14', '2021-06-21', '2021-06-28', 
         '2021-07-05', '2021-07-12', '2021-07-19', '2021-07-26', '2021-08-02', '2021-08-09', '2021-08-16',
         '2021-08-23', '2021-08-30', '2021-09-06', '2021-09-13', '2021-09-20', '2021-09-27', '2021-10-04', 
         '2021-10-11', '2021-10-18', '2021-10-25', '2021-11-01', '2021-11-08', '2021-11-15', '2021-11-22', 
         '2021-11-29', '2021-12-06', '2021-12-13', '2021-12-20', '2021-12-27', '2022-01-03', '2022-01-10', 
         '2022-01-17', '2022-01-24', '2022-01-31', '2022-02-07', '2022-02-14', '2022-02-21', '2022-02-28', 
         '2022-03-07', '2022-03-14', '2022-03-21', '2022-03-28', '2022-04-04', '2022-04-11', '2022-04-18',
         '2022-04-25', '2022-05-02', '2022-05-09', '2022-05-16', '2022-05-23', '2022-05-30', '2022-06-06', 
         '2022-06-13', '2022-06-20', '2022-06-27', '2022-07-04', '2022-07-11', '2022-07-18', '2022-07-25', 
         '2022-08-01', '2022-08-08', '2022-08-15', '2022-08-22', '2022-08-29', '2022-09-05', '2022-09-12', 
         '2022-09-19', '2022-09-26', '2022-10-03', '2022-10-10', '2022-10-17', '2022-10-24', '2022-10-31', 
         '2022-11-07', '2022-11-14', '2022-11-21', '2022-11-28', '2022-12-05', '2022-12-12', '2022-12-19',
         '2022-12-26', '2023-01-02', '2023-01-09', '2023-01-16', '2023-01-23', '2023-01-30', '2023-02-06', 
         '2023-02-13', '2023-02-20', '2023-02-27', '2023-03-06', '2023-03-13', '2023-03-20', '2023-03-27', 
         '2023-04-03', '2023-04-10', '2023-04-17', '2023-04-24', '2023-05-01', '2023-05-08', '2023-05-15', 
         '2023-05-22', '2023-05-29', '2023-06-05', '2023-06-12', '2023-06-26', '2023-07-03', '2023-07-10', 
         '2023-07-17', '2023-07-24', '2023-07-31', '2023-08-07', '2023-08-14', '2023-08-21', '2023-08-21', 
         '2023-08-28', '2023-09-04', '2023-09-11', '2023-09-18', '2023-09-25', '2023-10-02', '2023-10-09', 
         '2023-10-16', '2023-10-23', '2023-10-30', '2023-11-06', '2023-11-13',  '2023-11-20', '2023-11-27', 
         '2023-12-04', '2023-12-11', '2023-12-18', '2023-12-25', '2024-01-01', '2024-01-08', '2024-01-15', 
         '2024-01-22', '2024-01-29', '2024-02-05', '2024-02-12', '2024-02-19', '2024-02-26', '2024-03-04', 
         '2024-03-11', '2024-03-18', '2024-03-25', '2024-04-01', '2024-04-08', '2024-04-15', '2024-04-22', 
         '2024-04-29']

models = ['AIMdyn-SlidingKoopman', 'AIpert-pwllnod', 'AMM-EpiInvert', 'Auquan-SEIR', 'BPagano-RtDriven', 
         'CDDEP-SEIR_MCMC', 'CEID-Walk', 'CEPH-Rtrend_covid', 'CMU-TimeSeries', 'COVIDhub-4_week_ensemble',
         'COVIDhub-baseline', 'COVIDhub-ensemble', 'COVIDhub-trained_ensemble', 'COVIDhub_CDC-ensemble',
         'CU-nochange', 'CU-scenario_high', 'CU-scenario_low', 'CU-scenario_mid', 'CU-select', 
         'CUB_PopCouncil-SLSTM', 'CUBoulder-COVIDLSTM', 'CWRU-COVID_19Predict', 'Caltech-CS156', 'Columbia_UNC-SurvCon',
         'Covid19Sim-Simulator', 'CovidActNow-SEIR_CAN', 'CovidAnalytics-DELPHI', 'DDS-NBDS', 'FAIR-NRAR', 
         'FDANIHASU-Sweight', 'FRBSF_Wilson-Econometric', 'GT-DeepCOVID', 'GT_CHHS-COVID19', 'Geneva-DetGrowth',
         'Google_Harvard-CPF', 'HKUST-DNN', 'IBF-TimeSeries', 'IEM_MED-CovidProject', 'IHME-CurveFit', 
         'IQVIA_ACOE-STAN', 'ISUandPKU-vSEIdR', 'IUPUI-HkPrMobiDyR', 'Imperial-ensemble1', 'Imperial-ensemble2', 
         'IowaStateLW-STEM', 'JBUD-HMXK', 'JCB-PRM', 'JHUAPL-Bucky', 'JHUAPL-Gecko', 'JHUAPL-Morris', 
         'JHUAPL-SLPHospEns', 'JHUAPLTDWG-ICATTML', 'JHU_CSSE-DECOM', 'JHU_IDD-CovidSP', 'JHU_UNC_GAS-StatMechPool',
         'KITmetricslab-select_ensemble', 'Karlen-pypm', 'LANL-GrowthRate', 'LNQ-ens1', 'LUcompUncertLab-VAR_3streams',
         'LosAlamos_NAU-CModel_SDVaxVar', 'MIT-Cassandra', 'MITCovAlliance-SIR', 'MIT_CritData-GBCF',
         'MIT_ISOLAT-Mixtures', 'MOBS-GLEAM_COVID', 'MSRA-DeepST', 'MUNI-ARIMA', 'MUNI-VAR','Microsoft-DeepSTIA',
         'NCSU-COVSIM', 'NotreDame-FRED', 'NotreDame-mobility', 'OHT-nbx', 'OHT_JHU-nbxd', 'OliverWyman-Navigator',
         'OneQuietNight-ML', 'PR_UMD-CF_RepTiLe', 'PSI-DICE', 'PSI-DRAFT', 'PandemicCentral-COVIDForest',
         'PandemicCentral-USCounty', 'QJHong-Encounter', 'Quantori-Multiagents', 'RPI_UW-Mob_Collision',
         'RobertWalraven-ESG', 'SDSC_ISG-TrendModel', 'SGroup-RandomForest', 'STH-3PU', 'SWC-TerminusCM', 'SigSci-TS',
         'SteveMcConnell-CovidComplete', 'TTU-squider', 'UA-EpiCovDA', 'UCF-AEM', 'UCLA-SuEIR', 'UCM_MESALab-FoGSEIR',
         'UCSB-ACTS', 'UCSD_NEU-DeepGLEAM', 'UChicago-CovidIL', 'UChicago-CovidIL_100', 'UChicago-CovidIL_10_+',
         'UChicago-CovidIL_30_+', 'UChicago-CovidIL_40', 'UChicago-CovidIL_60', 'UChicago-CovidIL_80',
         'UChicagoCHATTOPADHYAY-UnIT', 'UMass-ExpertCrowd', 'UMass-MechBayes', 'UMass-gbq', 'UMass-sarix',
         'UMass-trends_ensemble', 'UMich-RidgeTfReg', 'USACE-ERDC_SEIR', 'USC-SI_kJalpha', 'USC-SI_kJalpha_RF',
         'USF-STPM', 'UT-Mobility', 'UT-Osiris', 'UT_GISAG-SPDM', 'UVA-Ensemble', 'UpstateSU-GRU', 
         'Wadhwani_AI-BayesOpt', 'WalmartLabsML-LogForecasting', 'YYG-ParamSearch', 'Yu_Group-CLEP',
         'cfa-wwrenewal', 'epiforecasts-ensemble1', 'prolix-euclidean']

forecasts = []
for model in models:
    for date in dates:
        try:
            predictions = pull_covid_early_predictions(model,date)
            predictions['team'] = model

            forecasts.append(predictions)
        except Exception as e:
            print(e)
covid_forecasts_early = pd.concat(forecasts, ignore_index=True)

In [ ]:
covid_forecasts_early.to_parquet('dat/covid19-forecast-hub_2020-2024.parquet')

# 2024-2025
https://github.com/CDCgov/covid19-forecast-hub/tree/main

In [ ]:
def pull_covid_predictions_2425(model,date):
    """ Load predictions of the model saved by the Covid forecast hub

    Parameters
    ----------
    model : str
        Model name on the
    dates : list or string
        List of potential dates in the iso format, e.g., 'yyyy-mm-dd', for the submission.
    """
    predictions = None
    
    url = f"https://raw.githubusercontent.com/CDCgov/covid19-forecast-hub/refs/heads/main/model-output/{model}/{date}-{model}"
    for ext in [".csv",'.parquet']:
        try:
            if ext == '.parquet':
                predictions = pd.read_parquet(url+ext,engine='auto')
                predictions['location'] = predictions['location'].astype(str)
                predictions['target_end_date'] = pd.to_datetime(predictions['target_end_date'])
            else:
                predictions = pd.read_csv(url+ext,dtype={'location':str},parse_dates=['target_end_date'])
        except:
            pass
    if predictions is None:
        print(f"Data for model {model} and date {date} unavailable")
    return predictions


In [ ]:
dates = ['2024-11-23', '2024-11-30', '2024-12-07', '2024-12-14', '2024-12-21', '2024-12-28', '2025-01-04', 
         '2025-01-11', '2025-01-18', '2025-02-01', '2025-02-08', '2025-02-15', '2025-02-22', '2025-03-01',
         '2025-03-08', '2025-03-15', '2025-03-22', '2025-03-29', '2025-04-05', '2025-04-12', '2025-04-19', 
         '2025-04-26', '2025-05-03', '2025-05-10', '2025-05-17', '2025-05-24', '2025-05-31', '2025-06-07', 
         '2025-06-14', '2025-06-21', '2025-06-28', '2025-07-05', '2025-07-12', '2025-07-19', '2025-07-26', 
         '2025-08-02', '2025-08-09', '2025-08-16', '2025-08-23', '2025-08-30', '2025-09-06', '2025-09-13',
         '2025-09-20', '2025-09-27', '2025-11-22', '2025-11-29', '2025-12-06', '2025-12-13', '2025-12-20',
         '2025-12-27']        
        

# all models from flusight github, not all will be used
models = ['CEPH-Rtrend_covid', 'CMU-TimeSeries', 'CFA_Pyrenew-Pyrenew_HE_COVID', 'CFA_Pyrenew-Pyrenew_HW_COVID',
          'CFA_Pyrenew-Pyrenew_H_COVID','CFA_Pyrenew-PyrenewHEW_COVID','CFA_Pyrenew-Pyrenew_E_COVID', 
          'Google_SAI-Ensemble', 'CMU-climate_baseline', 'CovidHub-baseline', 'CovidHub-ensemble', 
          'UGA_flucast-INFLAenza', 'JHU_CSSE-CSSE_Ensemble', 'MOBS-GLEAM_COVID', 'Metaculus-cp', 'OHT_JHU-nbxd', 
          'UM-DeepOutbreak', 'UMass-ar6_pooled', 'UMass-gbqr', 'NEU_ISI-AdaptiveEnsemble', 'CFA-EpiAutoGP',
          'CADPH-CovidCAT_Ensemble']

forecasts = []
for model in models:
    for date in dates:
        try:
            predictions = pull_covid_predictions_2425(model,date)
            predictions['team'] = model

            forecasts.append(predictions)
        except Exception as e:
            print(e)
            
covid_forecasts_2425 = pd.concat(forecasts, ignore_index=True)

In [ ]:
covid_forecasts_2425.to_parquet('dat/covid19-forecast-hub_2024-25.parquet')